<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Interior-point-methods-for-linear-programming" data-toc-modified-id="Interior-point-methods-for-linear-programming-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Interior point methods for linear programming</a></span><ul class="toc-item"><li><span><a href="#Part-I----Barrier-geometry" data-toc-modified-id="Part-I----Barrier-geometry-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>Part I -- Barrier geometry</a></span></li><li><span><a href="#Part-II----Central-path-and-certificate" data-toc-modified-id="Part-II----Central-path-and-certificate-1.2"><span class="toc-item-num">1.2&nbsp;&nbsp;</span>Part II -- Central path and certificate</a></span></li><li><span><a href="#Part-III----Path-following-IPM" data-toc-modified-id="Part-III----Path-following-IPM-1.3"><span class="toc-item-num">1.3&nbsp;&nbsp;</span>Part III -- Path-following IPM</a></span></li><li><span><a href="#Part-IV----What-the-line-search-does" data-toc-modified-id="Part-IV----What-the-line-search-does-1.4"><span class="toc-item-num">1.4&nbsp;&nbsp;</span>Part IV -- What the line search does</a></span></li><li><span><a href="#Part-V----Parameter-choices-on-a-higher-dimensional-benchmark" data-toc-modified-id="Part-V----Parameter-choices-on-a-higher-dimensional-benchmark-1.5"><span class="toc-item-num">1.5&nbsp;&nbsp;</span>Part V -- Parameter choices on a higher-dimensional benchmark</a></span></li></ul></li></ul></div>


# Interior point methods for linear programming -- student version

This practical work illustrates the interior-point method introduced in the lecture.
Parts I to IV use a single two-dimensional linear program:

$$
\min_x c^\top x \quad \text{s.t.} \quad Ax \le b.
$$

Keeping the same small LP makes the geometry visible: the feasible polygon, the slacks, the barrier, the central path, and the damped Newton steps can all be plotted.

Part V then switches to a 16-dimensional benchmark. The geometry is no longer visible, so the focus changes from visualization to algorithmic diagnostics: how many Newton systems are solved, how often the line search reduces the step, and how these quantities depend on `rho` and `N_inner`.

The gradient and Hessian formulas for the LP barrier are taken from the slides. The aim here is to connect those formulas with the practical path-following algorithm.


### Learning objectives

By the end of the practical work, you should be able to:

- interpret strict feasibility through the slack vector `b - Ax`;
- explain why the logarithmic barrier keeps iterates in the interior;
- visualize the central path and relate it to the certificate `nI/s`;
- describe the inner/outer structure of a path-following IPM;
- distinguish Newton linear solves from line-search reductions;
- compare `rho` and `N_inner` using explicit diagnostics.


In [ ]:
# Uncomment once if needed
# using Pkg
# Pkg.activate(".")
# Pkg.instantiate()
# Pkg.add(["Plots", "JuMP", "HiGHS"])


### Notebook setup

Use a Julia kernel.
The notebook uses `Plots` throughout, and the final benchmark uses `JuMP` with the `HiGHS` LP solver to compute a reference optimum.
`LinearAlgebra` and `Random` are standard libraries.

If one of the packages is not available in the notebook environment, run the setup cell below once, then restart the kernel.


### Data and mathematical helpers

The next cell defines the LP used in the geometric part of the practical work.
The constraints are written as `Ax <= b`, so the slack vector is

$$
r(x)=b-Ax.
$$

The helper functions follow the slide notation:

- `slacks(x)` computes `r(x)`;
- `is_strictly_feasible(x)` checks that all slacks are positive;
- `phi(x)` is the logarithmic barrier;
- `grad_phi(x)` and `hess_phi(x)` are the LP formulas from the slides;
- `barrier_objective(x, s)` is the penalized objective `s*c'x + phi(x)`.

The formulas are provided so that the notebook can focus on the algorithmic mechanism rather than on symbolic differentiation.


In [ ]:
using LinearAlgebra, Plots, Random

gr()
default(markerstrokewidth = 0, linewidth = 2, legend = :topright)
Random.seed!(1234)

# LP in inequality form: min c'x subject to Ax <= b.
A = [-1.0 0.0;
      0.0 -1.0;
      0.5 1.0;
      1.0 -1.0]
b = [0.0, 0.0, 1.0, 0.5]
c = [0.1, 1.0]
x_start = [0.4, 0.4]

nI, n = size(A)

slacks(x) = b - A * x
is_strictly_feasible(x; atol = 1e-10) = all(slacks(x) .> atol)

function phi(x)
    r = slacks(x)
    all(r .> 0) || return Inf
    return -sum(log, r)
end

# These are the LP barrier formulas from the slides.
grad_phi(x) = A' * (1.0 ./ slacks(x))
hess_phi(x) = A' * Diagonal((1.0 ./ slacks(x)) .^ 2) * A

function barrier_objective(x, s)
    ### TO COMPLETE:
    ### Return the barrier objective F_s(x) = s*c'x + phi(x).
    error("Complete barrier_objective")
end
barrier_gradient(x, s) = s * c + grad_phi(x)
barrier_hessian(x) = hess_phi(x)

function feasible_vertices(A, b; atol = 1e-8)
    m, n = size(A)
    vertices = Vector{Vector{Float64}}()
    for i in 1:m-1, j in i+1:m
        M = A[[i, j], :]
        if abs(det(M)) > atol
            x = M \ b[[i, j]]
            if all(A * x .<= b .+ atol) && !any(norm(x - y) <= 1e-7 for y in vertices)
                push!(vertices, x)
            end
        end
    end
    return vertices
end

vertices = feasible_vertices(A, b)
x_star = vertices[argmin([dot(c, v) for v in vertices])]
valP = dot(c, x_star)
println("LP optimum: x* = ", x_star, ", value = ", valP)


### Plotting helpers

The plotting helpers are not part of the algorithm. They only make the 2D example readable:
the polygon is the feasible set, the red point is the LP optimum, and the arrow shows the direction `-c` in which the linear objective decreases.


In [ ]:
function plot_lp(; title = "Feasible polygon")
    plt = plot(Shape([0, 0.5, 1, 0], [0, 0, 0.5, 1]);
        opacity = 0.25, color = :steelblue, label = "feasible set",
        xlabel = "x1", ylabel = "x2", aspect_ratio = :equal, title = title)
    scatter!(plt, [x_star[1]], [x_star[2]]; color = :crimson, markersize = 6, label = "LP optimum")
    plot!(plt, [x_start[1], x_start[1] - 0.12 * c[1]],
        [x_start[2], x_start[2] - 0.12 * c[2]]; arrow = 2, color = :black, label = "-c")
    return plt
end

function plot_points!(plt, pts; label = "", color = :orange, marker = :circle)
    plot!(plt, [p[1] for p in pts], [p[2] for p in pts];
        label = label, color = color, marker = marker)
end


### Newton and line-search helpers

The function `newton_step(x, s)` computes one damped Newton step for the barrier problem

$$
\min_x \; F_s(x) = s c^\top x + \phi(x).
$$

The raw Newton direction solves

$$
\nabla^2 F_s(x)\,d = -\nabla F_s(x).
$$

This direction is useful only if the resulting iterate remains inside the feasible set.
The function `backtracking_step` therefore first reduces `alpha` until `x + alpha*d` is strictly feasible.
It then applies Armijo's sufficient-descent test:

$$
F_s(x+\alpha d) \le F_s(x) + \eta\alpha\nabla F_s(x)^\top d.
$$

Since `d` is a descent direction, the right-hand side is smaller than `F_s(x)`.
The test prevents accepting a feasible step that gives too little decrease in the current barrier objective.


In [ ]:
function backtracking_step(x, dir, s; shrink = 0.5, armijo = 1e-4)
    alpha = 1.0
    n_boundary = 0
    while !is_strictly_feasible(x + alpha * dir)
        alpha *= shrink
        n_boundary += 1
    end

    fx = barrier_objective(x, s)
    g = barrier_gradient(x, s)
    n_armijo = 0
    while barrier_objective(x + alpha * dir, s) > fx + armijo * alpha * dot(g, dir)
        alpha *= shrink
        n_armijo += 1
    end
    return alpha, n_boundary + n_armijo
end

function newton_step(x, s)
    ### TO COMPLETE:
    ### 1. compute the barrier gradient and Hessian,
    ### 2. solve the Newton linear system for dir,
    ### 3. call backtracking_step,
    ### 4. return dir, alpha, and the number of backtracking reductions.
    error("Complete newton_step")
end

function center_at(s; x0 = x_start, tol = 1e-10, max_iter = 50)
    x = copy(x0)
    for _ in 1:max_iter
        norm(barrier_gradient(x, s)) <= tol && return x
        dir, alpha, _ = newton_step(x, s)
        x = x + alpha * dir
    end
    return x
end

function central_path(s_values; x0 = x_start)
    xs = Vector{Vector{Float64}}()
    x = copy(x0)
    for s in s_values
        x = center_at(s; x0 = x)
        push!(xs, copy(x))
    end
    return xs
end


## Part I -- Barrier geometry

We first inspect strict feasibility and the logarithmic barrier on the small LP.
For `Ax <= b`, the slack vector is

$$
r(x)=b-Ax.
$$

A point is strictly feasible when every component of `r(x)` is positive.
The logarithmic barrier is

$$
\phi(x)=-\sum_i \log(r_i(x)).
$$

It becomes very large near the boundary, which is why an interior-point method approaches the optimum from inside the feasible region.


In [ ]:
println("slacks at x_start = ", slacks(x_start))
println("strictly feasible? ", is_strictly_feasible(x_start))
plot_lp(title = "The 2D LP used throughout the notebook")


<div class="alert alert-success"> <b>Question 1)</b>

Evaluate the slacks at a few points of the polygon.
Which constraints are active at the LP optimum?
Why does the logarithmic barrier prevent an iterate from crossing the boundary?
</div>


<div class="alert alert-info" role="alert"><b>Question 1)</b>
Answer
</div>


## Part II -- Central path and certificate

For each `s > 0`, the barrier subproblem is

$$
\min_x \; s c^\top x + \phi(x).
$$

Its minimizer `x_s` is on the central path.
For this LP, the associated dual variables are

$$
\mu_i(s)=\frac{1}{s r_i(x_s)}.
$$

They give the optimality certificate

$$
c^\top x_s - \operatorname{val}(P) \le \frac{n_I}{s}.
$$

This part checks numerically that increasing `s` moves the central point toward the LP optimum and tightens the certificate.


<div class="alert alert-success"> <b>Question 2)</b>

Plot the central path for increasing values of `s`.
Compare the true gap `c'x_s - val(P)` with the certificate `nI/s`.
What does the table say when `s` becomes large?
</div>


In [ ]:
s_values = [1.0 * 1.35^k for k in 0:25]
x_path = central_path(s_values)

mu_path = [(1.0 ./ slacks(x)) ./ s for (x, s) in zip(x_path, s_values)]
actual_gap = [max(dot(c, x) - valP, 1e-14) for x in x_path]
certified_gap = [nI / s for s in s_values]

println("s        c'x_s       actual gap    nI/s")
for (s, x, gap, cert) in zip(s_values, x_path, actual_gap, certified_gap)
    println(rpad(round(s, digits = 3), 9), " ",
        lpad(round(dot(c, x), digits = 6), 10), " ",
        lpad(round(gap, digits = 6), 12), " ",
        lpad(round(cert, digits = 6), 10))
end

p_path = plot_lp(title = "Central path")
plot_points!(p_path, x_path; label = "central path", color = :blue, marker = :circle)

p_gap = plot(s_values, actual_gap;
    xscale = :log10, yscale = :log10, marker = :circle,
    xlabel = "s", ylabel = "optimality gap", label = "actual gap",
    title = "Gap and certificate")
plot!(p_gap, s_values, certified_gap; linestyle = :dash, label = "nI/s")

plot(p_path, p_gap, layout = (1, 2), size = (1050, 390))


<div class="alert alert-info" role="alert"><b>Question 2)</b>
Answer
</div>


## Part III -- Path-following IPM

The practical path-following algorithm does not solve each barrier subproblem from scratch.
It repeats two nested operations:

1. perform a fixed number `N_inner` of damped Newton steps for the current value of `s`;
2. increase the barrier parameter with `s <- rho*s`.

The next cells implement this loop and compare the generated iterates with the central path.


<div class="alert alert-success"> <b>Question 3)</b>

Before running the code, predict what should happen when `N_inner` is very small and when it is larger.
Then study the implementation of `path_following_ipm` and compare its trajectory with the central path.
What is the effect of increasing or decreasing `N_inner`?
</div>


In [ ]:
function path_following_ipm(x0, s0, rho; N_outer = 8, N_inner = 3)
    x = copy(x0)
    s = s0
    all_points = [copy(x)]
    outer_points = [copy(x)]
    steps = Float64[]

    for _ in 1:N_outer
        for _ in 1:N_inner
            ### TO COMPLETE:
            ### Compute one Newton step, update x, and store alpha and x.
            error("Complete the inner loop of path_following_ipm")
        end

        push!(outer_points, copy(x))

        ### TO COMPLETE:
        ### Update the barrier parameter for the next outer iteration.
        error("Complete the outer update of path_following_ipm")
    end

    return (all_points = all_points, outer_points = outer_points, steps = steps, x = x)
end


In [ ]:
out = path_following_ipm(x_start, 1.0, 2.0; N_outer = 10, N_inner = 3)
reference_path = central_path([1.0 * 1.15^k for k in 0:70])
plt = plot_lp(title = "Path-following trajectory")
plot_points!(plt, reference_path; label = "central path", color = :blue, marker = :none)
plot_points!(plt, out.all_points; label = "IPM iterates", color = :orange, marker = :cross)
scatter!(plt, [p[1] for p in out.outer_points], [p[2] for p in out.outer_points];
    color = :crimson, label = "outer iterates")
plt


<div class="alert alert-info" role="alert"><b>Question 3)</b>
Answer
</div>


## Part IV -- What the line search does

The line search has two roles:

1. keep `x + alpha*dir` strictly feasible;
2. enforce Armijo sufficient descent for the current barrier objective.

This part isolates the first role.
We compare two strictly feasible starting points for the same value of `s`:

- a well-centered point whose slacks are all reasonably large;
- a point very close to one boundary, with one very small slack.

Both starts are admissible. The question is whether being close to the boundary forces the first Newton steps to be shorter.


<div class="alert alert-success"> <b>Question 4)</b>

Run the comparison below.
Which start forces smaller accepted step sizes at the beginning?
How does this relate to the phrase “interior point”?
</div>


In [ ]:
function inner_trace(x0, s; n_steps = 6)
    x = copy(x0)
    alphas = Float64[]
    min_slacks = [minimum(slacks(x))]
    values = [barrier_objective(x, s)]
    for _ in 1:n_steps
        dir, alpha, _ = newton_step(x, s)
        x = x + alpha * dir
        push!(alphas, alpha)
        push!(min_slacks, minimum(slacks(x)))
        push!(values, barrier_objective(x, s))
    end
    return (alphas = alphas, min_slacks = min_slacks, values = values)
end

x_centered = [0.4, 0.4]
x_near_boundary = [0.0011354238621323898, 0.599599335692208]
s_trace = 100.0

tr_center = inner_trace(x_centered, s_trace)
tr_boundary = inner_trace(x_near_boundary, s_trace)

println("centered alphas      = ", tr_center.alphas)
println("near-boundary alphas = ", tr_boundary.alphas)

p1 = plot(tr_center.min_slacks; yscale = :log10, marker = :circle,
    label = "centered", xlabel = "inner iteration", ylabel = "minimum slack",
    title = "Distance to the boundary")
plot!(p1, tr_boundary.min_slacks; marker = :diamond, label = "near boundary")

p2 = plot(tr_center.alphas; marker = :circle, label = "centered",
    xlabel = "inner iteration", ylabel = "accepted alpha", title = "Accepted step sizes")
plot!(p2, tr_boundary.alphas; marker = :diamond, label = "near boundary")

plot(p1, p2, layout = (1, 2), size = (1050, 360))


<div class="alert alert-info" role="alert"><b>Question 4)</b>
Answer
</div>


## Part V -- Parameter choices on a higher-dimensional benchmark

The final part leaves the geometric example and uses a 16-dimensional LP with box constraints and several dense inequalities whose normals are close to parallel.
This benchmark is used to compare the algorithmic parameters `rho` and `N_inner`.

The reference optimum is computed with JuMP and the HiGHS LP solver.
The reported gap is therefore

$$
c^\top x - \operatorname{val}(P),
$$

up to the numerical tolerance of the external solver.

The comparison uses fixed `N_inner`, as in the slide pseudocode.
Every run starts from the same centered point at `s = 1`; this common initial centering is not counted.

Two quantities are recorded:

- `linear_solves`: the number of Newton systems solved, one per Newton direction;
- `backtracking reductions`: the number of times the line search reduces `alpha`.

Reducing `alpha` does not solve a new linear system. It still matters because many reductions indicate that the proposed Newton direction is poorly scaled for the current barrier problem.

In this LP notebook, evaluating slacks, gradients, and Hessians is simple matrix algebra.
In more general decision-focused or simulation-based settings, the analogous oracle calls may be much more expensive; that is why it is useful to record them explicitly.


<div class="alert alert-success"> <b>Question 5)</b>

Run the parameter grid on the larger benchmark.
Which `(rho, N_inner)` pair uses the fewest Newton linear solves to reach the target optimality gap?
Do the cheapest choices also have reasonable line-search behavior?
</div>


### Benchmark and reference solve

The following cells define the larger LP and solve it once with HiGHS.
The HiGHS optimum is used only as a reference value for the parameter comparison.


In [ ]:
using JuMP, HiGHS

function make_parameter_sensitivity_problem(; n = 16, n_dense = 5, seed = 88)
    rng = MersenneTwister(seed)

    # Dense inequalities with similar normals make the barrier subproblems less benign.
    base = exp.(range(-1.2, 1.2, length = n))
    dense_rows = Matrix{Float64}(undef, n_dense, n)
    for k in 1:n_dense
        perturb = 0.04 * randn(rng, n)
        dense_rows[k, :] = max.(base .* (1 .+ perturb), 1e-3)
    end

    # The starting point is strictly feasible by construction.
    x0_big = fill(0.62, n)
    demand = 0.45 .* (dense_rows * x0_big)

    # Constraints: 0 <= x <= 1 and dense_rows*x >= demand.
    A_big = vcat(-Matrix{Float64}(I, n, n), Matrix{Float64}(I, n, n), -dense_rows)
    b_big = vcat(zeros(n), ones(n), -demand)
    c_big = 0.15 .+ exp.(range(-0.3, 1.2, length = n)) .* (1 .+ 0.05 * randn(rng, n))

    return (A = A_big, b = b_big, c = c_big, x0 = x0_big, nI = size(A_big, 1))
end


In [ ]:
function solve_lp_with_highs(pb)
    n = length(pb.c)
    model = Model(HiGHS.Optimizer)
    set_silent(model)

    ### TO COMPLETE:
    ### Build the JuMP model for min pb.c'x subject to pb.A*x <= pb.b.
    ### Solve it with HiGHS, check that the status is optimal, and return
    ### (x = value.(x), value = objective_value(model), status = status).
    error("Complete solve_lp_with_highs")
end


### Barrier oracles for the benchmark

For this LP, the barrier value, gradient, and Hessian are simple matrix operations.
In a more complex model, these same oracle calls may be much more expensive.


In [ ]:
pb_slacks(pb, x) = pb.b - pb.A * x
pb_is_strictly_feasible(pb, x; atol = 1e-10) = all(pb_slacks(pb, x) .> atol)

function pb_phi(pb, x)
    r = pb_slacks(pb, x)
    all(r .> 0) || return Inf
    return -sum(log, r)
end

pb_barrier_objective(pb, x, s) = s * dot(pb.c, x) + pb_phi(pb, x)
pb_barrier_gradient(pb, x, s) = s * pb.c + pb.A' * (1.0 ./ pb_slacks(pb, x))
pb_barrier_hessian(pb, x) = pb.A' * Diagonal((1.0 ./ pb_slacks(pb, x)) .^ 2) * pb.A


### Newton step and line search

One call to `pb_barrier_newton_step` solves one Newton linear system.
The subsequent backtracking reductions only change the scalar step length `alpha`; they do not solve a new linear system.


In [ ]:
function pb_backtracking_to_boundary(pb, x, d; shrink = 0.5)
    alpha = 1.0
    n_back = 0

    # First enforce strict feasibility.
    while !pb_is_strictly_feasible(pb, x + alpha * d)
        alpha *= shrink
        n_back += 1
    end

    return alpha, n_back
end

function pb_barrier_newton_step(pb, x, s; armijo = 1e-4, shrink = 0.5)
    g = pb_barrier_gradient(pb, x, s)
    H = pb_barrier_hessian(pb, x)
    d = -(H \ g)  # This is the Newton linear solve counted in the experiment.

    alpha, n_back = pb_backtracking_to_boundary(pb, x, d; shrink = shrink)
    fx = pb_barrier_objective(pb, x, s)

    # Then enforce sufficient decrease of the current barrier objective.
    while pb_barrier_objective(pb, x + alpha * d, s) > fx + armijo * alpha * dot(g, d)
        alpha *= shrink
        n_back += 1
    end

    return d, alpha, n_back
end


In [ ]:
function pb_central_point(pb, s; x_start = pb.x0, tol = 1e-8, max_iter = 80)
    x = copy(x_start)
    for _ in 1:max_iter
        norm(pb_barrier_gradient(pb, x, s)) <= tol && return x
        d, alpha, _ = pb_barrier_newton_step(pb, x, s)
        x = x + alpha * d
    end
    return x
end


### Fixed-parameter path following

The next function is the actual comparison routine.
It runs path following with fixed `rho` and fixed `N_inner` until the HiGHS-based optimality gap is below the target.


In [ ]:
function fixed_inner_pb_until_gap(pb, x_centered, s0, rho, N_inner, opt_value;
        eps = 5e-3, max_outer = 80, max_s = 1e8)
    x = copy(x_centered)
    s = s0
    linear_solves = 0
    total_backtracks = 0
    min_alpha = 1.0
    best_gap = max(dot(pb.c, x) - opt_value, 0.0)

    for outer in 1:max_outer
        ### TO COMPLETE:
        ### 1. update s using rho,
        ### 2. perform N_inner Newton steps,
        ### 3. count linear solves and backtracking reductions separately,
        ### 4. update the best optimality gap,
        ### 5. return as soon as best_gap <= eps.
        error("Complete fixed_inner_pb_until_gap")
    end

    return (
        rho = rho,
        N_inner = N_inner,
        solved = false,
        outer_updates = max_outer,
        linear_solves = linear_solves,
        final_gap = best_gap,
        final_s = s,
        total_backtracks = total_backtracks,
        min_alpha = min_alpha,
    )
end


### Parameter grid

All runs start from the same centered point at `s = 1`.
The common centering phase is not counted in the comparison.


In [ ]:
pb = make_parameter_sensitivity_problem()
lp_solution = solve_lp_with_highs(pb)
x_centered_big = pb_central_point(pb, 1.0; x_start = pb.x0, tol = 1e-10, max_iter = 120)

rho_grid = [1.5, 2.0, 3.0, 4.0, 6.0]
N_inner_grid = [1, 2, 3, 5, 8]
target_gap = 5e-3

grid_rows = [
    fixed_inner_pb_until_gap(pb, x_centered_big, 1.0, rho, N_inner, lp_solution.value;
        eps = target_gap)
    for N_inner in N_inner_grid for rho in rho_grid
]


In [ ]:
println("HiGHS status = ", lp_solution.status)
println("HiGHS optimal value = ", round(lp_solution.value, digits = 6))
println("target optimality gap = ", target_gap)
println("rho    N_inner   solved   linear solves   final gap      backtracks   min alpha")

for row in grid_rows
    println(rpad(string(row.rho), 6), " ",
        lpad(string(row.N_inner), 7), " ",
        lpad(string(row.solved), 8), " ",
        lpad(string(row.linear_solves), 14), " ",
        lpad(string(round(row.final_gap, sigdigits = 4)), 12), " ",
        lpad(string(row.total_backtracks), 11), " ",
        lpad(string(round(row.min_alpha, sigdigits = 3)), 10))
end


### Diagnostic plots

The two heatmaps must be read together: the first counts Newton systems, while the second shows how often the line search had to shrink the step.


In [ ]:
linear_solve_matrix = reshape([row.linear_solves for row in grid_rows], length(rho_grid), length(N_inner_grid))'
backtrack_matrix = reshape([row.total_backtracks for row in grid_rows], length(rho_grid), length(N_inner_grid))'

p_solves = heatmap(
    string.(rho_grid),
    string.(N_inner_grid),
    linear_solve_matrix;
    xlabel = "rho",
    ylabel = "N_inner",
    colorbar_title = "linear solves",
    title = "Newton systems solved to target gap",
)

p_backtracks = heatmap(
    string.(rho_grid),
    string.(N_inner_grid),
    backtrack_matrix;
    xlabel = "rho",
    ylabel = "N_inner",
    colorbar_title = "reductions",
    title = "Line-search reductions",
)

plot(p_solves, p_backtracks, layout = (1, 2), size = (1120, 420))


<div class="alert alert-info" role="alert"><b>Question 5)</b>
Answer
</div>


<div class="alert alert-success"> <b>Question 6)</b>

Based on the experiments, what would you monitor in an implementation of an interior-point method besides the objective value?
</div>


<div class="alert alert-info" role="alert"><b>Question 6)</b>
Answer
</div>
